# Sprint 2 — Data Sampling
**Member A (Lawrence) · Computer Vision Team 8**

Samples FinTabNet into train/val/test splits and exports to Drive as CSV.
Run once — reuse every session by loading the CSV.

```
Cell 1  Mount Drive + unzip
Cell 2  Load labeled_df + build image index
Cell 3  Sampling controls (edit here)
Cell 4  Sample all types
Cell 5  Verify splits
Cell 6  Export to Drive
Cell 7  Reload and verify (use this every session)
```


## Cell 1 — Mount Drive + unzip

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import subprocess, pandas as pd, json, random

BASE     = Path('/content/drive/My Drive/Quarter 3/Computer Vision and DL/Computer_Vision_Team8')
ZIP_PATH = BASE / 'data' / 'hierarchical_tables_v1.zip'

EXTRACT_TO = Path('/content/data/processed')
EXTRACT_TO.mkdir(parents=True, exist_ok=True)

existing = list(EXTRACT_TO.rglob('*.png')) + list(EXTRACT_TO.rglob('*.jpg'))
if existing:
    print(f'Already extracted — {len(existing)} images.')
else:
    print('Unzipping...')
    subprocess.run(['unzip','-q',str(ZIP_PATH),'-d',str(EXTRACT_TO)], capture_output=True)
    imgs = list(EXTRACT_TO.rglob('*.png')) + list(EXTRACT_TO.rglob('*.jpg'))
    print(f'Done. {len(imgs)} images extracted.')

# Auto-detect image directory and JSONL
first_img     = next(EXTRACT_TO.rglob('*.png'), None)
IMG_DIR       = first_img.parent if first_img else EXTRACT_TO
jsonl_matches = list(EXTRACT_TO.rglob('hard_examples_subset.jsonl'))
JSONL_PATH    = jsonl_matches[0] if jsonl_matches else None

print(f'IMG_DIR    : {IMG_DIR}')
print(f'JSONL_PATH : {JSONL_PATH}')
print(f'JSONL exists: {JSONL_PATH is not None and JSONL_PATH.exists()}')


Mounted at /content/drive
Unzipping...
Done. 32670 images extracted.
IMG_DIR    : /content/data/processed/data/processed/images
JSONL_PATH : /content/data/processed/data/processed/hard_examples_subset.jsonl
JSONL exists: True


## Cell 2 — Load labeled_df + build image index

In [ ]:
# Load labeled_df — has imgid + image_type
labeled_df = pd.read_csv(BASE / 'outputs' / 'sprint1' / 'labeled dataset (df_combined).csv')
print(f'labeled_df: {len(labeled_df)} rows')
print(labeled_df['image_type'].value_counts().to_string())

# Load JSONL — has imgid + html_table (ground truth HTML)
print('\nLoading HTML from JSONL...')
img_id_to_html = {}
with open(JSONL_PATH) as f:
    for line in f:
        try:
            d = json.loads(line)
            if 'imgid' in d and 'html_table' in d:
                img_id_to_html[d['imgid']] = d['html_table']
        except: pass
print(f'HTML entries loaded: {len(img_id_to_html)}')

# Build image lookup — only files that exist on disk
print('Building image index...')
IMG_LOOKUP = {}
for p in IMG_DIR.glob('*.png'): IMG_LOOKUP[p.stem] = str(p)
for p in IMG_DIR.glob('*.jpg'): IMG_LOOKUP[p.stem] = str(p)
print(f'Images on disk: {len(IMG_LOOKUP)}')

# Find overlap — imgids with BOTH html and image on disk
common = set(img_id_to_html.keys()) & set(IMG_LOOKUP.keys())
print(f'Usable samples (html + image): {len(common)}')

# Filter labeled_df to matched only
matched_df = labeled_df[labeled_df['imgid'].isin(common)].copy()
print('\nUsable by type:')
print(matched_df['image_type'].value_counts().to_string())

labeled_df: 32670 rows
image_type
wide_table          15871
normal_table        11371
tall_table           3168
low_quality_blur     1320
low_contrast          940

Loading HTML from JSONL...
HTML entries loaded: 32670
Building image index...
Images on disk: 32670
Usable samples (html + image): 32670

Usable by type:
image_type
wide_table          15871
normal_table        11371
tall_table           3168
low_quality_blur     1320
low_contrast          940


## Cell 3 — Sampling controls
> **Edit numbers here only.**
> All downstream cells read from these variables.

### Strategy:
- Test: 200 per type (locked, FinTabNet only)
- Phase 1 pool: 700 per type (all 5 types)
- Phase 2 pool: 1,000 tall_table only (separate from Phase 1)
- Tall table sampling order: test → phase1 → phase2 → unused

In [ ]:
# ════════════════════════════════════════════════════════════
# SAMPLING CONTROLS — edit here only
# ════════════════════════════════════════════════════════════

RANDOM_SEED = 42   # never change — ensures reproducibility

# ── Test set (FinTabNet only, locked until Sprint 3) ────────
# 200 × 5 types = 1,000 test samples
TEST_PER_TYPE = 60

# ── Phase 1 pool (all types, used for general warm-up) ──────
# 700 × 5 types = 3,500 samples → 2,975 train + 525 val
PHASE1_PER_TYPE = 200

# ── Phase 2 pool (tall_table only, SEPARATE from Phase 1) ───
# 1,000 tall_table → 850 train + 150 val
# These are different samples from the 700 tall_table in Phase 1
PHASE2_TALL_COUNT = 0

# ── Val ratio (applied to both Phase 1 and Phase 2 pools) ───
VAL_RATIO = 0.15

# ── Output path on Drive (reused every session) ──────────────
SPLIT_SAVE_PATH = BASE / 'data' / 'training_fintabnet_pool_splits_1300sample.csv'
SPLIT_SAVE_PATH.parent.mkdir(parents=True, exist_ok=True)

# ════════════════════════════════════════════════════════════
# Validation guards
for img_type, count in matched_df['image_type'].value_counts().items():
    needed = TEST_PER_TYPE + PHASE1_PER_TYPE
    if img_type == 'tall_table':
        needed += PHASE2_TALL_COUNT
    if count < needed:
        print(f'WARNING: {img_type} has {count} samples but needs {needed}')
        print(f'  → reduce TEST_PER_TYPE or PHASE1_PER_TYPE')
    else:
        print(f'OK  {img_type:<20} available:{count:>6}  needed:{needed:>6}  unused:{count-needed:>6}')

print(f'\nTest total    : {TEST_PER_TYPE * 5}')
print(f'Phase1 pool   : {PHASE1_PER_TYPE * 5} → train {int(PHASE1_PER_TYPE*5*0.85)} / val {int(PHASE1_PER_TYPE*5*0.15)}')
print(f'Phase2 pool   : {PHASE2_TALL_COUNT} → train {int(PHASE2_TALL_COUNT*0.85)} / val {int(PHASE2_TALL_COUNT*0.15)}')


OK  wide_table           available: 15871  needed:   260  unused: 15611
OK  normal_table         available: 11371  needed:   260  unused: 11111
OK  tall_table           available:  3168  needed:   260  unused:  2908
OK  low_quality_blur     available:  1320  needed:   260  unused:  1060
OK  low_contrast         available:   940  needed:   260  unused:   680

Test total    : 300
Phase1 pool   : 1000 → train 850 / val 150
Phase2 pool   : 0 → train 0 / val 0


## Cell 4 — Sample all types
> Deterministic — same result every time with RANDOM_SEED=42.
> Tall table is sampled in order: test → phase1 → phase2 → unused.
> All other types: test → phase1 → unused.

In [ ]:
random.seed(RANDOM_SEED)

all_records = []  # will hold every sampled record with split label

for img_type in matched_df['image_type'].unique():
    # Shuffle this type with fixed seed for reproducibility
    type_df = matched_df[matched_df['image_type'] == img_type].copy()
    type_df = type_df.sample(frac=1, random_state=RANDOM_SEED).reset_index(drop=True)

    cursor = 0  # track position in shuffled list

    # ── Slice 1: Test ─────────────────────────────────────────────────────
    test_slice = type_df.iloc[cursor : cursor + TEST_PER_TYPE]
    cursor += TEST_PER_TYPE
    for _, row in test_slice.iterrows():
        all_records.append({
            'img_id'    : row['imgid'],
            'img_path'  : IMG_LOOKUP[row['imgid']],
            'html'      : img_id_to_html[row['imgid']],
            'image_type': img_type,
            'source'    : 'fintabnet',
            'phase'     : 'test',
            'split'     : 'test',
        })

    # ── Slice 2: Phase 1 pool ──────────────────────────────────────────────
    p1_slice = type_df.iloc[cursor : cursor + PHASE1_PER_TYPE]
    cursor += PHASE1_PER_TYPE
    n_p1_val   = max(1, round(len(p1_slice) * VAL_RATIO))
    n_p1_train = len(p1_slice) - n_p1_val
    for i, (_, row) in enumerate(p1_slice.iterrows()):
        split = 'val' if i < n_p1_val else 'train'
        all_records.append({
            'img_id'    : row['imgid'],
            'img_path'  : IMG_LOOKUP[row['imgid']],
            'html'      : img_id_to_html[row['imgid']],
            'image_type': img_type,
            'source'    : 'fintabnet',
            'phase'     : 'phase1',
            'split'     : split,
        })

    # ── Slice 3: Phase 2 pool (tall_table only, SEPARATE from Phase 1) ────
    if img_type == 'tall_table':
        p2_slice = type_df.iloc[cursor : cursor + PHASE2_TALL_COUNT]
        cursor += PHASE2_TALL_COUNT
        n_p2_val   = max(1, round(len(p2_slice) * VAL_RATIO))
        n_p2_train = len(p2_slice) - n_p2_val
        for i, (_, row) in enumerate(p2_slice.iterrows()):
            split = 'val' if i < n_p2_val else 'train'
            all_records.append({
                'img_id'    : row['imgid'],
                'img_path'  : IMG_LOOKUP[row['imgid']],
                'html'      : img_id_to_html[row['imgid']],
                'image_type': img_type,
                'source'    : 'fintabnet',
                'phase'     : 'phase2',
                'split'     : split,
            })

    used   = cursor
    unused = len(type_df) - used
    print(f'{img_type:<20} total:{len(type_df):>6}  used:{used:>6}  unused:{unused:>6}')

df_splits = pd.DataFrame(all_records)
print(f'\nTotal records sampled: {len(df_splits)}')


wide_table           total: 15871  used:   260  unused: 15611
tall_table           total:  3168  used:   260  unused:  2908
low_contrast         total:   940  used:   260  unused:   680
normal_table         total: 11371  used:   260  unused: 11111
low_quality_blur     total:  1320  used:   260  unused:  1060

Total records sampled: 1300


## Cell 5 — Verify splits

In [ ]:
print('=' * 60)
print('  SPLIT VERIFICATION')
print('=' * 60)

# ── Test set ──────────────────────────────────────────────────────────────
test_df = df_splits[df_splits['phase'] == 'test']
print(f'\nTEST SET ({len(test_df)} total) — FinTabNet only, locked until Sprint 3')
print(test_df['image_type'].value_counts().to_string())

# ── Phase 1 ───────────────────────────────────────────────────────────────
p1_df    = df_splits[df_splits['phase'] == 'phase1']
p1_train = p1_df[p1_df['split'] == 'train']
p1_val   = p1_df[p1_df['split'] == 'val']
print(f'\nPHASE 1 POOL ({len(p1_df)} total)')
print(f'  Train: {len(p1_train)}')
print(p1_train['image_type'].value_counts().to_string())
print(f'  Val  : {len(p1_val)}')
print(p1_val['image_type'].value_counts().to_string())


# ── Phase 2 ───────────────────────────────────────────────────────────────
p2_df    = df_splits[df_splits['phase'] == 'phase2']
p2_train = p2_df[p2_df['split'] == 'train']
p2_val   = p2_df[p2_df['split'] == 'val']
print(f'\nPHASE 2 POOL ({len(p2_df)} total — tall_table only, separate from Phase 1)')
print(f'  Train: {len(p2_train)}')
print(f'  Val  : {len(p2_val)}')


# ── Overlap check ─────────────────────────────────────────────────────────
test_ids   = set(test_df['img_id'])
p1_ids     = set(p1_df['img_id'])
p2_ids     = set(p2_df['img_id'])

print(f'\nOVERLAP CHECKS (all should be 0):')
print(f'  test ∩ phase1  : {len(test_ids & p1_ids)}')
print(f'  test ∩ phase2  : {len(test_ids & p2_ids)}')
print(f'  phase1 ∩ phase2: {len(p1_ids & p2_ids)}')

if len(test_ids & p1_ids) == 0 and len(test_ids & p2_ids) == 0 and len(p1_ids & p2_ids) == 0:
    print('\n✓ No overlap — splits are clean')
else:
    print('\n✗ OVERLAP DETECTED — check sampling logic')

# ── Summary ───────────────────────────────────────────────────────────────
print(f'\nSUMMARY:')
print(f'  Test          : {len(test_df):>5}  (locked)')
print(f'  Phase 1 train : {len(p1_train):>5}')
print(f'  Phase 1 val   : {len(p1_val):>5}')
print(f'  Phase 2 train : {len(p2_train):>5}  (tall_table only)')
print(f'  Phase 2 val   : {len(p2_val):>5}  (tall_table only)')
print(f'  TOTAL         : {len(df_splits):>5}')


  SPLIT VERIFICATION

TEST SET (300 total) — FinTabNet only, locked until Sprint 3
image_type
wide_table          60
tall_table          60
low_contrast        60
normal_table        60
low_quality_blur    60

PHASE 1 POOL (1000 total)
  Train: 850
image_type
wide_table          170
tall_table          170
low_contrast        170
normal_table        170
low_quality_blur    170
  Val  : 150
image_type
wide_table          30
tall_table          30
low_contrast        30
normal_table        30
low_quality_blur    30

PHASE 2 POOL (0 total — tall_table only, separate from Phase 1)
  Train: 0
  Val  : 0

OVERLAP CHECKS (all should be 0):
  test ∩ phase1  : 0
  test ∩ phase2  : 0
  phase1 ∩ phase2: 0

✓ No overlap — splits are clean

SUMMARY:
  Test          :   300  (locked)
  Phase 1 train :   850
  Phase 1 val   :   150
  Phase 2 train :     0  (tall_table only)
  Phase 2 val   :     0  (tall_table only)
  TOTAL         :  1300


## Cell 6 — Export to Drive
> Saves two files to Drive:
> 1. `dataset_splits.csv` — full split table with img_path (session-specific, rebuild each session)
> 2. `dataset_splits_ids.csv` — img_id + html + image_type + phase + split only (permanent, no paths)

> **Always reload from `dataset_splits_ids.csv` and rebuild img_path each session.**

In [ ]:
# ── Save full CSV (with img_path) ────────────────────────────────────────
# img_path is session-specific (/content/ resets every session)
# This file is for THIS session only
full_path = SPLIT_SAVE_PATH
df_splits.to_csv(full_path, index=False)
print(f'Full split CSV saved → {full_path}')

# ── Save permanent CSV (no img_path) ─────────────────────────────────────
# This file is permanent — reloaded every session and img_path rebuilt
permanent_cols = ['img_id', 'html', 'image_type', 'source', 'phase', 'split']
perm_path = SPLIT_SAVE_PATH.parent / 'dataset_splits_ids_1300sample.csv'
df_splits[permanent_cols].to_csv(perm_path, index=False)
print(f'Permanent split CSV saved → {perm_path}')
print(f'(img_path not saved — rebuilt each session from IMG_LOOKUP)')

# ── Print file info ───────────────────────────────────────────────────────
print(f'\nFiles saved:')
print(f'  {full_path.name:<35} {full_path.stat().st_size/1e3:.1f} KB')
print(f'  {perm_path.name:<35} {perm_path.stat().st_size/1e3:.1f} KB')
print(f'\nColumns in permanent CSV: {permanent_cols}')
print(f'\nSample rows:')
print(df_splits[permanent_cols].head(3).to_string(index=False))

Full split CSV saved → /content/drive/My Drive/Quarter 3/Computer Vision and DL/Computer_Vision_Team8/data/training_fintabnet_pool_splits_1300sample.csv
Permanent split CSV saved → /content/drive/My Drive/Quarter 3/Computer Vision and DL/Computer_Vision_Team8/data/dataset_splits_ids_1300sample.csv
(img_path not saved — rebuilt each session from IMG_LOOKUP)

Files saved:
  training_fintabnet_pool_splits_1300sample.csv 4227.8 KB
  dataset_splits_ids_1300sample.csv   4140.7 KB

Columns in permanent CSV: ['img_id', 'html', 'image_type', 'source', 'phase', 'split']

Sample rows:
          img_id                                                                                                                                                                                                                                                                                                                                                                                                                   

## Cell 7 — Reload and verify
> **Use this cell at the start of every training session.**
> Loads the permanent CSV from Drive and rebuilds img_path for current session.
> Run Cells 1-2 first (mount Drive + build IMG_LOOKUP), then run this cell.

In [ ]:
# ── Load permanent split CSV ─────────────────────────────────────────────
perm_path = BASE / 'data' / 'training_fintabnet_pool_splits_1300sample.csv'

if not perm_path.exists():
    raise FileNotFoundError(
        f'Split CSV not found at {perm_path}\n'
        'Run this notebook from Cell 1 to generate the splits first.'
    )

df_loaded = pd.read_csv(perm_path)
print(f'Loaded {len(df_loaded)} rows from {perm_path.name}')

# ── Rebuild img_path from current session IMG_LOOKUP ─────────────────────
df_loaded['img_path'] = df_loaded['img_id'].map(IMG_LOOKUP)
missing = df_loaded['img_path'].isna().sum()
if missing > 0:
    print(f'WARNING: {missing} images not found in current session.')
    print('Did Cell 1 unzip complete? Check IMG_LOOKUP.')
    df_loaded = df_loaded[df_loaded['img_path'].notna()].reset_index(drop=True)
    print(f'Continuing with {len(df_loaded)} valid samples.')
else:
    print(f'✓ All {len(df_loaded)} image paths rebuilt successfully.')

# ── Convert to sample lists for training ─────────────────────────────────
def to_sample_list(df):
    """Convert DataFrame rows to list of dicts for Dataset class."""
    return df[['img_id','img_path','html','image_type','source']].to_dict('records')

# Test set — locked
test_samples = to_sample_list(
    df_loaded[df_loaded['phase'] == 'test']
)

# Phase 1 — all types, train + val
phase1_train = to_sample_list(
    df_loaded[(df_loaded['phase']=='phase1') & (df_loaded['split']=='train')]
)
phase1_val = to_sample_list(
    df_loaded[(df_loaded['phase']=='phase1') & (df_loaded['split']=='val')]
)

# Phase 2 — tall_table only, train + val (separate from Phase 1)
phase2_train = to_sample_list(
    df_loaded[(df_loaded['phase']=='phase2') & (df_loaded['split']=='train')]
)
phase2_val = to_sample_list(
    df_loaded[(df_loaded['phase']=='phase2') & (df_loaded['split']=='val')]
)

# ── Print summary ─────────────────────────────────────────────────────────
print(f'\nReady for training:')
print(f'  test_samples   : {len(test_samples):>5}  (locked — Sprint 3 only)')
print(f'  phase1_train   : {len(phase1_train):>5}')
print(f'  phase1_val     : {len(phase1_val):>5}')
print(f'  phase2_train   : {len(phase2_train):>5}  (tall_table only)')
print(f'  phase2_val     : {len(phase2_val):>5}  (tall_table only)')

print(f'\nPhase 1 type breakdown (train):')
p1_train_df = df_loaded[(df_loaded['phase']=='phase1') & (df_loaded['split']=='train')]
print(p1_train_df['image_type'].value_counts().to_string())

print(f'\nSample record (phase1 train):')
if phase1_train:
    s = phase1_train[0]
    print(f'  img_id     : {s["img_id"]}')
    print(f'  img_path   : {s["img_path"]}')
    print(f'  image_type : {s["image_type"]}')
    print(f'  html preview: {s["html"][:80]}')
    from pathlib import Path
    print(f'  file exists: {Path(s["img_path"]).exists()}')


Loaded 1300 rows from training_fintabnet_pool_splits_1300sample.csv
✓ All 1300 image paths rebuilt successfully.

Ready for training:
  test_samples   :   300  (locked — Sprint 3 only)
  phase1_train   :   850
  phase1_val     :   150
  phase2_train   :     0  (tall_table only)
  phase2_val     :     0  (tall_table only)

Phase 1 type breakdown (train):
image_type
wide_table          170
tall_table          170
low_contrast        170
normal_table        170
low_quality_blur    170

Sample record (phase1 train):
  img_id     : fintabnet_013536
  img_path   : /content/data/processed/data/processed/images/fintabnet_013536.png
  image_type : wide_table
  html preview: <table>
 <tr>
  <td>
  </td>
  <td colspan="2">
   2009
  </td>
  <td colspan="2
  file exists: True


## Cell 8 — Quick view of split distribution

In [ ]:
import pandas as pd

print('Full split distribution:')
summary = df_splits.groupby(['phase','split','image_type']).size().reset_index(name='count')
pivot   = summary.pivot_table(index=['phase','split'], columns='image_type', values='count', fill_value=0)
pivot['TOTAL'] = pivot.sum(axis=1)
print(pivot.to_string())

print('\nPhase summary:')
phase_summary = df_splits.groupby(['phase','split']).size().reset_index(name='count')
for _, row in phase_summary.iterrows():
    print(f'  {row["phase"]:<10} {row["split"]:<8} {row["count"]:>5}')


Full split distribution:
image_type    low_contrast  low_quality_blur  normal_table  tall_table  wide_table  TOTAL
phase  split                                                                             
phase1 train         170.0             170.0         170.0       170.0       170.0  850.0
       val            30.0              30.0          30.0        30.0        30.0  150.0
test   test           60.0              60.0          60.0        60.0        60.0  300.0

Phase summary:
  phase1     train      850
  phase1     val        150
  test       test       300


In [ ]:
print('Null values in df_splits (training_fintabnet_pool_splits.csv):')
print(df_splits.isnull().sum())

Null values in df_splits (training_fintabnet_pool_splits.csv):
img_id        0
img_path      0
html          0
image_type    0
source        0
phase         0
split         0
dtype: int64
